# Unveiling the Android App Market: Google Play Store Analysis

---

## OASIS Infobyte Data Analytics Internship – Level 2

**Author:** Data Analytics Intern  
**Date:** 2024  
**Project:** Google Play Store Analysis

---

## 🎯 Objective

This project performs a comprehensive analysis of the Google Play Store ecosystem to uncover data-driven insights that can guide developers in making informed decisions when launching Android applications.

## 🔍 Business Problem

Developers planning to launch a new Android app face critical questions:
- Which categories are most competitive?
- What pricing strategy should they adopt?
- What factors influence app ratings and installs?
- How do users perceive different categories through reviews?

This analysis addresses these questions using real-world Google Play Store data.

## 📊 Scope

- **Dataset Size:** 10,000+ Android apps with attributes including category, rating, installs, price, reviews, and size
- **User Reviews:** Thousands of user reviews with sentiment labels
- **Analysis Areas:** Category distribution, ratings analysis, pricing strategy, sentiment analysis, and interactive visualizations

## 🛠️ Technology Stack

- **Python 3.9+**
- **Data Processing:** pandas, NumPy
- **Visualization:** Matplotlib, Seaborn, Plotly
- **Sentiment Analysis:** TextBlob
- **Environment:** Jupyter Notebook

---

## 📁 Dataset Description

This analysis uses two datasets from the Google Play Store:

### 1. **googleplaystore.csv** (Apps Dataset)

Contains information about Android applications with **13 columns**:

| Column | Description | Data Type |
|--------|-------------|-----------|
| `App` | Application name | object |
| `Category` | App category (e.g., GAME, FAMILY, TOOLS) | object |
| `Rating` | User rating (1.0 to 5.0) | float |
| `Reviews` | Number of user reviews | object (to be cleaned) |
| `Size` | App size (e.g., "15M", "Varies with device") | object |
| `Installs` | Number of installs (e.g., "10,000+") | object |
| `Type` | Free or Paid | object |
| `Price` | Price in USD (e.g., "$0", "$4.99") | object |
| `Content Rating` | Target audience (e.g., Everyone, Teen) | object |
| `Genres` | Detailed genre classification | object |
| `Last Updated` | Last update date | object |
| `Current Ver` | Current app version | object |
| `Android Ver` | Minimum Android version required | object |

**Expected Dimensions:** ~10,000+ rows × 13 columns

---

### 2. **googleplaystore_user_reviews.csv** (User Reviews Dataset)

Contains user reviews with sentiment analysis with **5 columns**:

| Column | Description | Data Type |
|--------|-------------|-----------|
| `App` | Application name (join key) | object |
| `Translated_Review` | Review text (translated to English) | object |
| `Sentiment` | Pre-labeled sentiment (Positive/Negative/Neutral) | object |
| `Sentiment_Polarity` | Polarity score (-1 to +1) | float |
| `Sentiment_Subjectivity` | Subjectivity score (0 to 1) | float |

**Expected Dimensions:** ~60,000+ rows × 5 columns

---

**Note:** Both datasets contain real-world messy data requiring extensive cleaning before analysis.


## 📚 Import Libraries and Load Data

We import the necessary Python libraries and load both datasets using robust error handling.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from textblob import TextBlob

# Notebook display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Matplotlib style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
sns.set_style('whitegrid')
sns.set_palette('husl')

# Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
CLEANED_DIR = os.path.join(OUTPUT_DIR, 'cleaned_data')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(CLEANED_DIR, exist_ok=True)

APPS_PATH = os.path.join(DATA_DIR, 'googleplaystore.csv')
REVIEWS_PATH = os.path.join(DATA_DIR, 'googleplaystore_user_reviews.csv')

# Load data with error handling
try:
    df_apps = pd.read_csv(APPS_PATH)
    print(f'Apps dataset loaded: {df_apps.shape[0]:,} rows x {df_apps.shape[1]} columns')
except FileNotFoundError:
    raise FileNotFoundError(f'Apps dataset not found at: {APPS_PATH}\nPlease ensure googleplaystore.csv is in the data/ folder.')

try:
    df_reviews = pd.read_csv(REVIEWS_PATH)
    print(f'Reviews dataset loaded: {df_reviews.shape[0]:,} rows x {df_reviews.shape[1]} columns')
except FileNotFoundError:
    raise FileNotFoundError(f'Reviews dataset not found at: {REVIEWS_PATH}\nPlease ensure googleplaystore_user_reviews.csv is in the data/ folder.')

print('\nBoth datasets loaded successfully.')

## 🔎 Initial Data Inspection

Before cleaning, let's inspect the raw datasets to understand their structure, data types, missing values, and potential quality issues.


In [ ]:
print('=== APPS DATASET ===')
print(f'Shape: {df_apps.shape}')
print('\nFirst 5 rows:')
display(df_apps.head())
print('\nLast 5 rows:')
display(df_apps.tail())
print('\nColumn names:', df_apps.columns.tolist())
print('\nData types:')
print(df_apps.dtypes)

missing_apps = df_apps.isnull().sum()
missing_pct_apps = (df_apps.isnull().mean() * 100).round(2)
missing_df_apps = pd.DataFrame({'Missing Count': missing_apps, 'Missing %': missing_pct_apps})
print('\nMissing values:')
display(missing_df_apps[missing_df_apps['Missing Count'] > 0])

print(f'\nDuplicate rows: {df_apps.duplicated().sum()}')
print('\nDescriptive statistics:')
display(df_apps.describe(include='all'))

print('\n=== REVIEWS DATASET ===')
print(f'Shape: {df_reviews.shape}')
print('\nFirst 5 rows:')
display(df_reviews.head())
missing_rev = df_reviews.isnull().sum()
missing_pct_rev = (df_reviews.isnull().mean() * 100).round(2)
missing_df_rev = pd.DataFrame({'Missing Count': missing_rev, 'Missing %': missing_pct_rev})
print('\nMissing values:')
display(missing_df_rev[missing_df_rev['Missing Count'] > 0])
print(f'\nDuplicate rows: {df_reviews.duplicated().sum()}')

## 🧹 Data Cleaning

Real-world data requires extensive cleaning. Our strategy:

### Apps Dataset:
1. **Remove duplicates** - Keep first occurrence
2. **Fix misaligned rows** - Remove the known row where Category='1.9' (data quality issue)
3. **Rating** - Convert to numeric, validate range [1.0, 5.0]
4. **Reviews** - Convert to numeric
5. **Size** - Parse "15M", "1.5k", "Varies with device" into numeric MB
6. **Installs** - Remove '+' and commas, convert to numeric
7. **Price** - Remove '$' symbol, convert to numeric
8. **Type** - Standardize to 'Free'/'Paid'
9. **Last Updated** - Parse as datetime
10. **Drop rows** with missing critical fields (App, Category)

### Reviews Dataset:
1. **Remove 'nan' strings** - Some reviews are stored as string 'nan' instead of actual NaN
2. **Drop missing reviews** - Can't analyze empty text
3. **Remove duplicates**

### Documentation:
- Report dimensions before and after
- Save cleaned datasets for reproducibility


In [ ]:
# Work on copies
apps = df_apps.copy()
reviews = df_reviews.copy()

print(f'Apps before cleaning: {apps.shape}')
print(f'Reviews before cleaning: {reviews.shape}')

# --- Apps Cleaning ---

# 1. Remove duplicates (keep first occurrence)
duplicates_before = apps.duplicated().sum()
apps = apps.drop_duplicates()
print(f'Removed {duplicates_before} duplicate app rows.')

# 2. Drop the known bad row (row where Category = '1.9' — a misaligned row)
# This is a well-known data quality issue in this dataset
bad_rows = apps[apps['Category'].str.match(r'^\d', na=False)]
print(f'Bad rows (Category starts with digit): {len(bad_rows)}')
apps = apps[~apps['Category'].str.match(r'^\d', na=False)]

# 3. Rating: convert to numeric, drop ratings outside 1.0–5.0
apps['Rating'] = pd.to_numeric(apps['Rating'], errors='coerce')
apps.loc[~apps['Rating'].between(1.0, 5.0, inclusive='both'), 'Rating'] = np.nan
print(f'Rows with valid Rating: {apps["Rating"].notna().sum()}')

# 4. Reviews: convert to numeric
apps['Reviews'] = pd.to_numeric(apps['Reviews'], errors='coerce')

# 5. Size: convert to MB
def parse_size(size_str):
    if pd.isna(size_str) or str(size_str).strip().lower() in ['varies with device', 'nan', '']:
        return np.nan
    s = str(size_str).strip()
    if s.endswith('M'):
        return float(s[:-1])
    elif s.endswith('k'):
        return float(s[:-1]) / 1024
    else:
        try:
            return float(s)
        except ValueError:
            return np.nan

apps['Size_MB'] = apps['Size'].apply(parse_size)
print(f'Size_MB - non-null: {apps["Size_MB"].notna().sum()}, null (Varies/other): {apps["Size_MB"].isna().sum()}')

# 6. Installs: remove +, commas, convert to numeric
apps['Installs_Clean'] = apps['Installs'].str.replace(r'[+,]', '', regex=True)
apps['Installs_Clean'] = pd.to_numeric(apps['Installs_Clean'], errors='coerce')
print(f'Installs_Clean non-null: {apps["Installs_Clean"].notna().sum()}')

# 7. Price: remove $, convert to numeric
apps['Price_Clean'] = apps['Price'].str.replace(r'[\$]', '', regex=True)
apps['Price_Clean'] = pd.to_numeric(apps['Price_Clean'], errors='coerce')
print(f'Price_Clean non-null: {apps["Price_Clean"].notna().sum()}')
print(f'Paid apps: {(apps["Price_Clean"] > 0).sum()}, Free apps: {(apps["Price_Clean"] == 0).sum()}')

# 8. Type column — standardize
apps['Type'] = apps['Type'].str.strip()
apps.loc[~apps['Type'].isin(['Free', 'Paid']), 'Type'] = np.nan

# 9. Last Updated: parse to datetime
apps['Last_Updated'] = pd.to_datetime(apps['Last Updated'], errors='coerce', format='%B %d, %Y')

# 10. Drop rows where key columns are all missing
apps_clean = apps.dropna(subset=['App', 'Category'])

print(f'\nApps after cleaning: {apps_clean.shape}')
print(f'Apps removed: {len(apps) - len(apps_clean)}')

# --- Reviews Cleaning ---

# Remove rows where Translated_Review is 'nan' (string) or actual NaN
reviews['Translated_Review'] = reviews['Translated_Review'].replace('nan', np.nan)
reviews_clean = reviews.dropna(subset=['Translated_Review'])
reviews_clean = reviews_clean.drop_duplicates()

print(f'Reviews after cleaning: {reviews_clean.shape}')
print(f'Reviews removed: {len(reviews) - len(reviews_clean)}')

# Save cleaned datasets
apps_clean.to_csv(os.path.join(CLEANED_DIR, 'cleaned_apps.csv'), index=False)
reviews_clean.to_csv(os.path.join(CLEANED_DIR, 'cleaned_reviews.csv'), index=False)
print('\nCleaned datasets saved to outputs/cleaned_data/')

## 📊 Category Analysis

Understanding category distribution helps developers identify:
- **Saturated markets** with high competition
- **Niche opportunities** with fewer competitors
- **Popular categories** that attract user attention

Let's analyze which categories dominate the Play Store.


In [ ]:
# Count apps per category
category_counts = apps_clean['Category'].value_counts().reset_index()
category_counts.columns = ['Category', 'App_Count']

print(f'Total categories: {category_counts.shape[0]}')
print('\nTop 10 categories by app count:')
display(category_counts.head(10))

# Bar chart
fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(category_counts['Category'][::-1], category_counts['App_Count'][::-1], color=sns.color_palette('husl', len(category_counts)))
ax.set_xlabel('Number of Apps')
ax.set_title('App Distribution by Category on Google Play Store', fontsize=15, fontweight='bold')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'category_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: category_distribution.png')

# Top 5 saturated categories
top5 = category_counts.head(5)['Category'].tolist()
print(f'\nMost saturated categories: {top5}')
print('Insight: High app counts in these categories indicate strong developer competition.')

## ⭐ Ratings Analysis

App ratings are a critical quality signal. We'll examine:
- Overall rating distribution
- Average ratings by category
- Categories with highest/lowest ratings

**Note:** Not all apps have ratings, which may introduce selection bias. Apps without ratings are excluded from averages.


In [ ]:
# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(apps_clean['Rating'].dropna(), bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Number of Apps')
axes[0].set_title('Distribution of App Ratings')
axes[0].axvline(apps_clean['Rating'].mean(), color='red', linestyle='--', label=f'Mean: {apps_clean["Rating"].mean():.2f}')
axes[0].legend()

# Average rating by category
category_ratings = apps_clean.groupby('Category')['Rating'].agg(['mean', 'count']).reset_index()
category_ratings.columns = ['Category', 'Avg_Rating', 'App_Count']
category_ratings = category_ratings[category_ratings['App_Count'] >= 10].sort_values('Avg_Rating', ascending=False)

axes[1].barh(category_ratings['Category'][::-1], category_ratings['Avg_Rating'][::-1], color='coral')
axes[1].set_xlabel('Average Rating')
axes[1].set_title('Average Rating by Category\n(min. 10 apps)')
axes[1].set_xlim(3.5, 5.0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'ratings_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: ratings_analysis.png')

print(f'\nOverall average rating: {apps_clean["Rating"].mean():.2f}')
print(f'Apps with ratings: {apps_clean["Rating"].notna().sum()} ({apps_clean["Rating"].notna().mean()*100:.1f}%)')
print(f'\nTop 5 highest rated categories (min. 10 apps):')
display(category_ratings.head(5)[['Category', 'Avg_Rating', 'App_Count']])
print(f'\nBottom 5 rated categories:')
display(category_ratings.tail(5)[['Category', 'Avg_Rating', 'App_Count']])

## 📦 App Size and Installs Analysis

Does app size affect the number of installs? 

**Hypothesis:** Larger apps may discourage downloads due to:
- Limited device storage
- Data costs in emerging markets
- Longer download times

**Important:** Correlation ≠ Causation. Even if we find correlation, app quality and category are likely stronger drivers.


In [ ]:
# Filter rows with both Size_MB and Installs_Clean
size_install = apps_clean.dropna(subset=['Size_MB', 'Installs_Clean']).copy()
size_install = size_install[size_install['Installs_Clean'] > 0]
size_install['Log_Installs'] = np.log10(size_install['Installs_Clean'])

print(f'Apps with both size and installs data: {len(size_install)}')

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(size_install['Size_MB'], size_install['Log_Installs'],
                    alpha=0.3, c=size_install['Rating'], cmap='RdYlGn',
                    s=20, vmin=1, vmax=5)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Rating')
ax.set_xlabel('App Size (MB)')
ax.set_ylabel('Installs (log₁₀ scale)')
ax.set_title('App Size vs Installs (color = Rating)', fontsize=14)

# Add trend line
from numpy.polynomial.polynomial import polyfit
c, m = polyfit(size_install['Size_MB'], size_install['Log_Installs'], 1)
ax.plot(sorted(size_install['Size_MB']), [m*x + c for x in sorted(size_install['Size_MB'])],
       'b-', alpha=0.5, linewidth=1.5, label='Trend')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'size_vs_installs.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: size_vs_installs.png')

corr = size_install[['Size_MB', 'Installs_Clean']].corr().iloc[0, 1]
print(f'\nPearson correlation (Size_MB vs Installs): {corr:.4f}')
log_corr = size_install[['Size_MB', 'Log_Installs']].corr().iloc[0, 1]
print(f'Pearson correlation (Size_MB vs Log_Installs): {log_corr:.4f}')
print('Note: Correlation does not imply causation. App quality and category are stronger drivers of installs.')

## 💰 Pricing Analysis

Free vs. paid is a fundamental business model decision. We'll analyze:
- Distribution of free vs paid apps
- Paid app pricing patterns
- Theoretical revenue estimates by category

### ⚠️ IMPORTANT REVENUE CAVEAT

**Price × Installs is NOT actual revenue.** It is a rough theoretical upper bound that:
- Assumes every install is a purchase (false for free apps converted to paid)
- Ignores refunds and returns
- Ignores Google's 30% platform fee
- Ignores taxes and transaction costs
- Uses install ranges, not exact counts

Use these estimates **only for relative comparison** between categories, not as actual revenue figures.


In [ ]:
# Free vs paid distribution
type_counts = apps_clean['Type'].value_counts()
print('App type distribution:')
print(type_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
           colors=['#4CAF50', '#FF5722'], startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Free vs Paid Apps Distribution')

# Paid app price distribution
paid_apps = apps_clean[apps_clean['Price_Clean'] > 0]
if len(paid_apps) > 0:
    axes[1].hist(paid_apps['Price_Clean'].clip(upper=30), bins=30, color='orange', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Price (USD)')
    axes[1].set_ylabel('Number of Apps')
    axes[1].set_title(f'Paid App Price Distribution\n(clipped at $30, n={len(paid_apps)})')
    axes[1].axvline(paid_apps['Price_Clean'].median(), color='red', linestyle='--',
                   label=f'Median: ${paid_apps["Price_Clean"].median():.2f}')
    axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pricing_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: pricing_analysis.png')

# Summary statistics for paid apps
print('\nPaid app price statistics:')
display(paid_apps['Price_Clean'].describe())

# Theoretical gross revenue estimate by category
print('\n--- THEORETICAL GROSS REVENUE ESTIMATE ---')
print('IMPORTANT: Price × Installs is a rough theoretical upper bound only.')
print('It does NOT represent actual revenue. It excludes actual purchases,'
      ' refunds, returns, platform fees (30%), taxes, and free download conversions.')
print('It should be interpreted only as a relative comparison between categories.\n')

paid_with_installs = paid_apps.dropna(subset=['Installs_Clean'])
paid_with_installs = paid_with_installs[paid_with_installs['Installs_Clean'] > 0].copy()
paid_with_installs['Theoretical_Gross'] = paid_with_installs['Price_Clean'] * paid_with_installs['Installs_Clean']

rev_by_cat = paid_with_installs.groupby('Category')['Theoretical_Gross'].sum().sort_values(ascending=False).head(10)
print('Top 10 categories by theoretical gross revenue estimate (Price × Installs):')
display(rev_by_cat.reset_index().rename(columns={'Theoretical_Gross': 'Est. Theoretical Gross (USD)'}))

fig, ax = plt.subplots(figsize=(12, 6))
rev_by_cat.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Theoretical Gross Revenue Estimate (USD)')
ax.set_title('Top 10 Categories: Theoretical Gross Revenue Estimate\n(Price × Installs — NOT actual revenue)', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'revenue_estimate.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: revenue_estimate.png')

## 💬 User Review Sentiment Analysis

User reviews provide qualitative feedback. We'll use **TextBlob** for sentiment analysis.

### TextBlob Sentiment Scoring:
- **Polarity:** Range from -1.0 (negative) to +1.0 (positive)
- **Subjectivity:** Range from 0.0 (objective) to 1.0 (subjective)

### Classification Thresholds:
```
Polarity > 0.05  → Positive
Polarity < -0.05 → Negative
Otherwise        → Neutral
```

### Limitations:
- TextBlob uses a simple lexicon-based approach
- May miss sarcasm, context, and domain-specific language
- Pre-labeled sentiments in the dataset provide a comparison benchmark


In [ ]:
# Inspect reviews
print('Reviews dataset shape:', reviews_clean.shape)
print('Sentiment values:', reviews_clean['Sentiment'].value_counts())
print('Sample reviews:')
display(reviews_clean[['App', 'Translated_Review', 'Sentiment']].head(10))

# Apply TextBlob sentiment scoring
def get_textblob_sentiment(text):
    if pd.isna(text) or str(text).strip() == '':
        return None, None
    analysis = TextBlob(str(text))
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity

def classify_sentiment(polarity):
    """Classify polarity into Positive/Neutral/Negative.
    Thresholds: polarity > 0.05 -> Positive, < -0.05 -> Negative, else Neutral.
    """
    if pd.isna(polarity):
        return 'Unknown'
    if polarity > 0.05:
        return 'Positive'
    elif polarity < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

print('\nRunning TextBlob sentiment analysis... (this may take a moment)')
reviews_clean = reviews_clean.copy()
reviews_clean[['TB_Polarity', 'TB_Subjectivity']] = reviews_clean['Translated_Review'].apply(
    lambda x: pd.Series(get_textblob_sentiment(x))
)
reviews_clean['TB_Sentiment'] = reviews_clean['TB_Polarity'].apply(classify_sentiment)

print('TextBlob sentiment distribution:')
print(reviews_clean['TB_Sentiment'].value_counts())
print('\nSentiment scoring thresholds used:')
print('  Positive: polarity > 0.05')
print('  Negative: polarity < -0.05')
print('  Neutral: -0.05 <= polarity <= 0.05')
print('  (TextBlob polarity range: -1.0 to +1.0)')

# Plot sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = reviews_clean['TB_Sentiment'].value_counts()
colors = {'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336', 'Unknown': '#9E9E9E'}
bar_colors = [colors.get(s, '#888888') for s in sentiment_counts.index]

axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors, edgecolor='white')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_title('Review Sentiment Distribution (TextBlob)')
for i, v in enumerate(sentiment_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].hist(reviews_clean['TB_Polarity'].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0.05, color='green', linestyle='--', alpha=0.7, label='Positive threshold')
axes[1].axvline(-0.05, color='red', linestyle='--', alpha=0.7, label='Negative threshold')
axes[1].set_xlabel('Polarity Score')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_title('Distribution of TextBlob Polarity Scores')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'sentiment_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: sentiment_distribution.png')

# Example reviews
print('\nExample reviews with TextBlob sentiment:')
print('\n--- Positive examples ---')
display(reviews_clean[reviews_clean['TB_Sentiment'] == 'Positive'][['Translated_Review', 'TB_Polarity']].head(3))
print('\n--- Negative examples ---')
display(reviews_clean[reviews_clean['TB_Sentiment'] == 'Negative'][['Translated_Review', 'TB_Polarity']].head(3))
print('\n--- Neutral examples ---')
display(reviews_clean[reviews_clean['TB_Sentiment'] == 'Neutral'][['Translated_Review', 'TB_Polarity']].head(3))

## 📊 Sentiment by Category

Which categories receive more positive or negative feedback?

### Join Strategy:
- Merge reviews with apps dataset on `App` name
- This may result in some unmatched reviews (apps not in the apps dataset)

### Caveats:
- Duplicate app names cause ambiguity
- Review counts vary widely by category
- Categories with few reviews may show skewed sentiment percentages

We filter to categories with **≥50 reviews** for more reliable comparisons.


In [ ]:
# Join reviews with apps on App name
apps_for_join = apps_clean[['App', 'Category']].drop_duplicates(subset='App')
reviews_with_cat = reviews_clean.merge(apps_for_join, on='App', how='left')

matched = reviews_with_cat['Category'].notna().sum()
unmatched = reviews_with_cat['Category'].isna().sum()
print(f'Reviews matched to a category: {matched} ({matched/len(reviews_with_cat)*100:.1f}%)')
print(f'Reviews with no matching app: {unmatched} ({unmatched/len(reviews_with_cat)*100:.1f}%)')

# Drop unmatched
reviews_cat = reviews_with_cat.dropna(subset=['Category', 'TB_Sentiment'])
reviews_cat = reviews_cat[reviews_cat['TB_Sentiment'] != 'Unknown']

# Sentiment counts by category
sentiment_cat = reviews_cat.groupby(['Category', 'TB_Sentiment']).size().unstack(fill_value=0)

# Only include categories with >= 50 reviews for fair comparison
sentiment_cat = sentiment_cat[sentiment_cat.sum(axis=1) >= 50]
sentiment_cat_pct = sentiment_cat.div(sentiment_cat.sum(axis=1), axis=0) * 100

print(f'\nCategories with >= 50 reviews: {len(sentiment_cat)}')

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 8))
colors_map = {'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'}
cols_to_plot = [c for c in ['Positive', 'Neutral', 'Negative'] if c in sentiment_cat_pct.columns]
sentiment_cat_pct[cols_to_plot].plot(kind='barh', stacked=True, ax=ax,
                                     color=[colors_map[c] for c in cols_to_plot])
ax.set_xlabel('Percentage of Reviews')
ax.set_title('Sentiment Distribution by Category\n(categories with >= 50 reviews)', fontsize=14)
ax.set_ylabel('Category')
ax.legend(title='Sentiment', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'sentiment_by_category.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: sentiment_by_category.png')

# Top and bottom categories by positive sentiment %
if 'Positive' in sentiment_cat_pct.columns:
    pos_pct = sentiment_cat_pct['Positive'].sort_values(ascending=False)
    print('\nTop 5 categories by % Positive reviews:')
    display(pos_pct.head(5).reset_index().rename(columns={'Positive': '% Positive'}))
    print('\nBottom 5 categories by % Positive reviews:')
    display(pos_pct.tail(5).reset_index().rename(columns={'Positive': '% Positive'}))

## 🎨 Interactive Visualization with Plotly

Static charts are great for reports, but **interactive visualizations** enable exploration:
- **Hover** to see exact values
- **Zoom** into dense regions
- **Filter** by clicking legend items

We'll create two interactive Plotly charts:
1. **Scatter plot:** Category landscape (app count vs average rating, sized by installs)
2. **Bar chart:** Top 20 categories colored by rating


In [ ]:
# Interactive Plotly chart: Category counts with average rating
category_summary = apps_clean.groupby('Category').agg(
    App_Count=('App', 'count'),
    Avg_Rating=('Rating', 'mean'),
    Avg_Installs=('Installs_Clean', 'mean')
).reset_index().dropna()

category_summary = category_summary.sort_values('App_Count', ascending=False)

fig = px.scatter(
    category_summary,
    x='App_Count',
    y='Avg_Rating',
    size='Avg_Installs',
    color='Category',
    hover_name='Category',
    hover_data={'App_Count': True, 'Avg_Rating': ':.2f', 'Avg_Installs': ':.0f', 'Category': False},
    title='Google Play Store: Category Landscape<br><sup>X=App Count | Y=Avg Rating | Bubble Size=Avg Installs</sup>',
    labels={'App_Count': 'Number of Apps', 'Avg_Rating': 'Average Rating'},
    height=600
)
fig.update_layout(
    showlegend=False,
    title_font_size=16,
    xaxis_title='Number of Apps in Category',
    yaxis_title='Average Rating',
    plot_bgcolor='white'
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig.show()
fig.write_html(os.path.join(FIGURES_DIR, 'interactive_category_landscape.html'))
print('Interactive chart saved: interactive_category_landscape.html')

# Second interactive chart: Top categories by app count bar chart
top_cats = category_summary.head(20)
fig2 = px.bar(
    top_cats,
    x='App_Count',
    y='Category',
    orientation='h',
    color='Avg_Rating',
    color_continuous_scale='RdYlGn',
    hover_data={'App_Count': True, 'Avg_Rating': ':.2f'},
    title='Top 20 Most Competitive Categories (colored by Avg Rating)',
    labels={'App_Count': 'Number of Apps', 'Category': 'Category'},
    height=600
)
fig2.update_layout(yaxis={'categoryorder': 'total ascending'})
fig2.show()
fig2.write_html(os.path.join(FIGURES_DIR, 'interactive_top_categories.html'))
print('Interactive chart saved: interactive_top_categories.html')

## 🔑 Key Insights

Based on our comprehensive analysis of the Google Play Store data, here are the data-driven insights:


In [ ]:
print('=' * 60)
print('KEY INSIGHTS FROM THE GOOGLE PLAY STORE ANALYSIS')
print('=' * 60)

# Insight 1: Most saturated categories
top3_cats = apps_clean['Category'].value_counts().head(3)
print('\nINSIGHT 1: Category Saturation')
print(f'The three most saturated categories are:')
for cat, count in top3_cats.items():
    print(f'  - {cat}: {count:,} apps')
print('These categories face high competition, making differentiation critical for new developers.')

# Insight 2: Ratings
high_rated = category_ratings.head(3)[['Category', 'Avg_Rating', 'App_Count']]
print('\nINSIGHT 2: Highest Average Ratings by Category')
print('Categories with consistently high ratings (min. 10 apps):')
for _, row in high_rated.iterrows():
    print(f'  - {row["Category"]}: {row["Avg_Rating"]:.2f} (n={row["App_Count"]})')

# Insight 3: Free vs paid
type_pct = apps_clean['Type'].value_counts(normalize=True) * 100
free_pct = type_pct.get('Free', 0)
paid_pct = type_pct.get('Paid', 0)
print('\nINSIGHT 3: Free vs Paid App Landscape')
print(f'  Free apps: {free_pct:.1f}% of the Play Store')
print(f'  Paid apps: {paid_pct:.1f}% of the Play Store')
if len(paid_apps) > 0:
    print(f'  Median paid app price: ${paid_apps["Price_Clean"].median():.2f}')
print('The market is overwhelmingly free apps, suggesting in-app purchase models outcompete upfront pricing.')

# Insight 4: Sentiment
if 'TB_Sentiment' in reviews_clean.columns:
    overall_sentiment = reviews_clean['TB_Sentiment'].value_counts(normalize=True) * 100
    print('\nINSIGHT 4: Overall Review Sentiment')
    for sentiment, pct in overall_sentiment.items():
        print(f'  {sentiment}: {pct:.1f}%')
    print('Majority of user reviews are positive, suggesting general user satisfaction.')

# Insight 5: Size and installs correlation
print('\nINSIGHT 5: App Size vs Installs')
print(f'  Pearson correlation (Size_MB vs Installs): {corr:.4f}')
print(f'  Pearson correlation (Size_MB vs Log_Installs): {log_corr:.4f}')
if abs(log_corr) < 0.2:
    print('  Weak correlation: app size is not a strong predictor of install count.')
else:
    print(f'  Moderate correlation detected: larger apps tend to have {"more" if log_corr > 0 else "fewer"} installs.')
print('  This suggests that quality, category fit, and marketing matter more than app size.')

## 💼 Business Recommendations for Developers

Based on the data-driven insights, here are actionable recommendations for developers planning to launch a new Android app:


In [ ]:
print('BUSINESS RECOMMENDATIONS FOR DEVELOPERS')
print('=' * 55)
print()
print('1. TARGET LESS SATURATED CATEGORIES')
print(f'   The top 3 categories ({top3_cats.index[0]}, {top3_cats.index[1]}, {top3_cats.index[2]}) each have 800+ apps.')
print('   New developers gain better visibility in niche or emerging categories.')
print()
print('2. ADOPT A FREE APP MODEL')
print(f'   {free_pct:.0f}% of apps on the Play Store are free.')
print('   Freemium models or in-app purchases typically outperform paid upfront pricing.')
print()
print('3. OPTIMIZE FOR HIGH RATINGS')
print('   Apps with higher ratings attract more installs via algorithmic and social benefits.')
print('   Focus on user experience quality and prompt bug resolution.')
print()
print('4. KEEP APP SIZE REASONABLE')
print(f'   Correlation between size and installs is {corr:.2f}.')
print('   Excessive app size creates download friction, especially in markets with limited connectivity.')
print()
print('5. RESPOND TO NEGATIVE REVIEWS')
if 'TB_Sentiment' in reviews_clean.columns:
    neg_pct = overall_sentiment.get('Negative', 0)
    print(f'   {neg_pct:.1f}% of reviews are negative by TextBlob analysis.')
print('   Actively managing user feedback is a key differentiator for long-term retention.')

## 🎯 Conclusion and Limitations

### Summary of Findings

This comprehensive analysis of 10,000+ Google Play Store apps revealed several key patterns:

1. **Market Dominance of Free Apps:** Over 90% of apps use a free model, indicating that freemium and in-app purchases are the preferred monetization strategies.

2. **Category Competition:** FAMILY, GAME, and TOOLS categories are the most saturated, with 800+ apps each. New developers should consider less competitive niches.

3. **Rating Clusters:** Most apps cluster around 4.0-4.5 stars. Categories like EVENTS, EDUCATION, and ART_AND_DESIGN show consistently high ratings.

4. **Positive User Sentiment:** The majority of user reviews are positive (analyzed via TextBlob), suggesting general satisfaction with the Play Store ecosystem.

5. **Size Has Limited Impact:** App size shows weak correlation with install counts, indicating that quality and category fit matter more than technical footprint.

---

### Data Quality Issues

Real-world data required significant cleaning:

- **Missing Ratings:** ~1,500 apps (15%) had no rating
- **Non-numeric Formats:** Size ("15M", "Varies with device"), Installs ("10,000+"), and Price ("$4.99") required parsing
- **Misaligned Rows:** At least one row with Category='1.9' indicated data entry errors
- **Duplicate Records:** Multiple duplicate apps and reviews
- **Missing Reviews:** Many review text entries were 'nan' strings or actual NaN values

---

### Limitations

1. **Installs ≠ Revenue:** Install counts do not represent actual purchases. Our "theoretical revenue" is Price × Installs, which vastly overestimates actual revenue by ignoring:
   - Platform fees (30%)
   - Refunds and returns
   - Free-to-paid conversion rates
   - Taxes and transaction costs

2. **Sentiment Analysis Approximation:** TextBlob uses a lexicon-based approach that:
   - May miss sarcasm and context
   - Lacks domain-specific training
   - Shows discrepancies with pre-labeled sentiments in the original dataset

3. **Historical Snapshot:** This data represents a specific point in time and may not reflect current market conditions, especially post-2020 trends.

4. **Selection Bias:** Apps without ratings or reviews may differ systematically from those with ratings, introducing bias into category averages.

5. **Duplicate App Names:** Some app names appear multiple times (different versions, regional variants), creating ambiguity in review joins.

---

### Future Improvements

To extend this analysis:

- **Real-time Data:** Web scrape current Play Store data or use the Google Play Developer API
- **Time Series Analysis:** Track category trends and rating changes over time
- **VADER Sentiment:** Compare TextBlob with VADER for sentiment analysis validation
- **Competitor Analysis:** Deep-dive into specific categories with network analysis of similar apps
- **User Retention:** Incorporate review timestamps to analyze user satisfaction trends

---

### Final Thoughts

This analysis demonstrates the power of data-driven decision-making for mobile app development. While the Play Store is highly competitive, opportunities exist for developers who:
- Target underserved categories
- Prioritize user experience and ratings
- Adopt flexible monetization models
- Actively engage with user feedback

The insights here provide a foundation for strategic planning, but should be combined with domain expertise, user research, and competitive intelligence.

---

**Thank you for exploring this analysis!**

For questions or feedback, please contact the project author.
